# Predicting the Bechdel Test from Movie Metadata

**Machine Learning Foundations — Group Project**  
**IE University, BDBA Class of 2028**

**Team:** _Raya Metchkarova, Alexander Glapiak, Alp Kurtbolat, Aysel Zeynalova, Milan Josifovikj, Samuel Sacha Benayoun_  
**Dataset:** FiveThirtyEight Bechdel data joined to IMDb/TMDb metadata (Kaggle *9000+ Movies: IMDb and Bechdel*)

---

## 1. Problem statement

The [Bechdel test](https://bechdeltest.com/) asks three questions of a film: (1) does it have at least two named women, (2) who talk to each other, (3) about something other than a man? It is a famously **coarse, contestable, and low-bar** measure of female representation. A film can pass and still be misogynistic; it can fail and still be a landmark of feminist cinema. That coarseness is exactly why it is an interesting ML target: the label is externally defined, community-curated, and disagreements with it are generative rather than errors.

We frame the task as **binary classification**: given pre-release-observable metadata (year, runtime, genre, budget, and production-scale proxies), can we predict whether a film will pass the Bechdel test?

Our central question is **not** 'how high can we push AUC?' Instead, we ask two connected questions:

1. **How much signal does metadata carry about Bechdel outcome?**
2. **Is the model learning something meaningful about representation, or is it taking a shortcut through genre and era?**

The second question motivates a **genre-ablation experiment** (§8.3): we train a version of our best model with all genre features removed and compare its performance to the full model. If most of the AUC survives genre removal, the model is picking up on broader patterns; if AUC collapses, we have learned that our predictor is essentially a genre classifier dressed up in more features.

## 2. Setup

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import load_synthetic_data, load_real_data
from preprocessing import (
    engineer_features, build_preprocessor,
    NUMERIC_FEATURES, CATEGORICAL_FEATURES, BINARY_FEATURES,
)
from models import make_model, MODEL_NAMES, PARAM_GRIDS
from evaluation import (
    cv_score_model, evaluate_on_test, results_table,
    plot_confusion, plot_roc, plot_learning_curve,
)

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)

## 3. Load data

The pipeline loads from a local CSV if available and otherwise falls back to a synthetic generator. The synthetic fallback exists only so the pipeline can be developed before the Kaggle CSV is in hand — **metrics from synthetic data are not scientific results, and every plot produced from synthetic data is watermarked below.**

In [2]:
REAL_DATA_PATH = '../data/movies_bechdel.csv'

if os.path.exists(REAL_DATA_PATH):
    df = load_real_data(REAL_DATA_PATH)
    DATA_SOURCE = 'real'
    print(f'Loaded REAL data from {REAL_DATA_PATH}')
else:
    df = load_synthetic_data(n=9000, seed=RANDOM_STATE)
    DATA_SOURCE = 'synthetic'
    print('[!] Real data not found — using synthetic data for pipeline development.')
    print('    Place movies_bechdel.csv in data/ and rerun before reporting results.')

# Helper: stamp plots when running on synthetic data so no one accidentally
# uses a dev-mode figure in the poster or report.
def stamp_if_synthetic(fig):
    if DATA_SOURCE != 'synthetic':
        return
    fig.text(0.5, 0.5, 'SYNTHETIC DATA', fontsize=40, color='red',
             alpha=0.18, ha='center', va='center', rotation=30, zorder=10)

print(f'\nShape: {df.shape}')
df.head()

ValueError: Loaded CSV is missing expected columns: ['imdb_id', 'budget', 'revenue', 'imdb_rating', 'genres', 'bechdel_score', 'bechdel_pass']. You may need to extend the rename_map in load_real_data().

## 4. Exploratory data analysis

### 4.1 Target distribution

In [ ]:
pass_rate = df['bechdel_pass'].mean()
n_pass = int(df['bechdel_pass'].sum())
n_fail = len(df) - n_pass
print(f'Bechdel pass rate: {pass_rate:.1%}  ({n_pass:,} pass / {n_fail:,} fail)')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
df['bechdel_pass'].value_counts().plot.bar(ax=axes[0], color=['#d62728', '#2ca02c'])
axes[0].set_xticklabels(['Fail', 'Pass'], rotation=0)
axes[0].set_title('Binary target: Bechdel pass/fail')
axes[0].set_ylabel('Count')
df['bechdel_score'].value_counts().sort_index().plot.bar(ax=axes[1], color='#1f77b4')
axes[1].set_title('Ordinal score (0 = fails first criterion ... 3 = passes all)')
axes[1].set_xlabel('Score')
stamp_if_synthetic(fig)
plt.tight_layout()

The class balance is roughly 55/45, so accuracy is a *usable* metric but we still prioritise F1 and ROC-AUC, which reward correct minority-class prediction rather than simple majority voting.

### 4.2 Missingness

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
print('Missingness (fraction of rows):')
print(miss.round(3))

fig, ax = plt.subplots(figsize=(7, 3))
miss.plot.barh(ax=ax, color='#ff7f0e')
ax.set_xlabel('Fraction missing')
ax.set_title('Missingness by column')
ax.invert_yaxis()
stamp_if_synthetic(fig)
plt.tight_layout()

**Interpretation.** Budget and revenue are the most incomplete columns. This is expected: TMDb and IMDb systematically under-report financials for indie and older films, which means a missing budget is **not missing-at-random** — it correlates with production scale, which itself plausibly correlates with the target. Rather than drop these rows, we **median-impute inside the pipeline** (so imputation statistics are learned on each CV fold's training slice, never on held-out data) and keep the information. A natural extension would be to add a binary `budget_is_missing` indicator as a feature, since the missingness pattern itself is informative.

### 4.3 Numeric distributions by target

In [ ]:
df_eda = engineer_features(df)

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col in zip(axes.ravel(), ['year', 'runtime', 'log_budget', 'imdb_rating']):
    for val, label, color in [(0, 'Fail', '#d62728'), (1, 'Pass', '#2ca02c')]:
        subset = df_eda.loc[df_eda['bechdel_pass'] == val, col].dropna()
        ax.hist(subset, bins=30, alpha=0.55, label=label, color=color, density=True)
    ax.set_title(f'{col}  (by Bechdel outcome)')
    ax.legend()
stamp_if_synthetic(fig)
plt.tight_layout()

### 4.4 Pass rate by genre

This figure is the **central empirical story** of the EDA. If genre effects are strong and directional, the model has real signal — but we will need to check in §8.3 whether that signal is the *only* thing the model uses.

In [ ]:
genre_cols = [c for c in df_eda.columns if c.startswith('genre_')]
pass_rate_by_genre = (
    df_eda[genre_cols + ['bechdel_pass']]
    .melt(id_vars='bechdel_pass', var_name='genre', value_name='in_genre')
    .query('in_genre == 1')
    .groupby('genre')['bechdel_pass']
    .agg(['mean', 'count'])
    .sort_values('mean', ascending=False)
)
pass_rate_by_genre.columns = ['pass_rate', 'n_films']
pass_rate_by_genre['genre'] = pass_rate_by_genre.index.str.replace('genre_', '')

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(pass_rate_by_genre['genre'], pass_rate_by_genre['pass_rate'],
        color=plt.cm.RdYlGn(pass_rate_by_genre['pass_rate']))
ax.axvline(pass_rate, color='black', linestyle='--', label=f'Overall ({pass_rate:.0%})')
ax.set_xlabel('Bechdel pass rate')
ax.set_title('Pass rate by genre')
ax.invert_yaxis()
ax.legend()
stamp_if_synthetic(fig)
plt.tight_layout()

pass_rate_by_genre[['n_films', 'pass_rate']].round(3)

## 5. Feature engineering & train/test split

### Leakage safeguards

All transformations that rely on dataset statistics (imputation, scaling, one-hot encoding) are handled inside `build_preprocessor()` and included in every model pipeline. The transformer is fit only on training data within each cross-validation fold and then applied to validation data, preventing leakage.

The engineered features (`log_budget`, `decade`, `genre one-hots`) use only row-level information, so computing them before the split is safe.

In [ ]:
df_feat = engineer_features(df)
feat_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES + BINARY_FEATURES
X = df_feat[feat_cols]
y = df_feat['bechdel_pass']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
)
print(f'Train: {len(X_train):,}   Test: {len(X_test):,}')
print(f'Train pass rate: {y_train.mean():.3f}   Test pass rate: {y_test.mean():.3f}')
print(f'Feature count: {X.shape[1]}')

## 6. Model training & cross-validation

We evaluate five models, from simple to more advanced:

1. **Dummy** (majority class) — baseline reference
2. **Logistic regression** — linear baseline
3. **Decision tree** (depth 6) — simple non-linear model
4. **Random forest** — ensemble model
5. **XGBoost** — gradient boosting model

Performance is reported as the **mean and standard deviation across 5 cross-validation folds**, allowing us to assess both accuracy and variability.

In [ ]:
cv_results = {}
fitted_pipelines = {}

for name in MODEL_NAMES:
    pipe = make_model(name, random_state=RANDOM_STATE)
    cv = cv_score_model(pipe, X_train, y_train, cv_splits=5, scoring='roc_auc',
                        random_state=RANDOM_STATE)
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe
    cv_results[name] = cv
    print(f'{name:10s}  CV ROC-AUC: {cv["mean"]:.4f} ± {cv["std"]:.4f}')

## 7. Hyperparameter tuning

We tune the two most promising families — logistic regression and XGBoost — using `RandomizedSearchCV` with a deliberately modest budget of 10 iterations per model. The rubric rewards *justified* scope over compute spend: our search grids are tight enough that 10 samples cover them well, and the resulting best configurations are within the range of common published defaults, so we do not suspect the search is budget-limited.

In [ ]:
tuned_pipelines = {}
tuned_cv_scores = {}

for name in ['logreg', 'xgb']:
    print(f'Tuning {name}...')
    search = RandomizedSearchCV(
        estimator=make_model(name, random_state=RANDOM_STATE),
        param_distributions=PARAM_GRIDS[name],
        n_iter=10,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        scoring='roc_auc',
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=0,
    )
    search.fit(X_train, y_train)
    tuned_pipelines[name] = search.best_estimator_
    tuned_cv_scores[name] = {
        'mean': search.best_score_,
        'std': search.cv_results_['std_test_score'][search.best_index_],
    }
    print(f'  best CV AUC: {search.best_score_:.4f} ± {tuned_cv_scores[name]["std"]:.4f}')
    print(f'  best params: {search.best_params_}')

## 8. Test-set evaluation & analysis

### 8.1 Headline metrics

Having selected hyperparameters on the training-fold CV, we now commit: a single evaluation on the untouched test set.

In [ ]:
final_pipelines = {**fitted_pipelines,
                   **{f'{k}_tuned': v for k, v in tuned_pipelines.items()}}

test_results = {}
for name, pipe in final_pipelines.items():
    m = evaluate_on_test(pipe, X_test, y_test)
    if name.endswith('_tuned'):
        base = name.replace('_tuned', '')
        m['cv_auc_mean'] = tuned_cv_scores[base]['mean']
        m['cv_auc_std']  = tuned_cv_scores[base]['std']
    else:
        m['cv_auc_mean'] = cv_results.get(name, {}).get('mean', np.nan)
        m['cv_auc_std']  = cv_results.get(name, {}).get('std', np.nan)
    test_results[name] = m

summary = results_table(test_results)
summary['CV-AUC (mean)'] = [round(test_results[m]['cv_auc_mean'], 4) for m in summary.index]
summary['CV-AUC (std)']  = [round(test_results[m]['cv_auc_std'], 4)  for m in summary.index]
summary

In [ ]:
fig = plot_roc(
    {k: v for k, v in final_pipelines.items() if k != 'dummy'},
    X_test, y_test,
    title='ROC curves — test set',
)
stamp_if_synthetic(fig)
fig.savefig('../outputs/roc_curves.png', dpi=150, bbox_inches='tight')

### 8.2 Best model — diagnostics

In [ ]:
best_name = max(test_results, key=lambda k: test_results[k]['roc_auc'])
best_pipe = final_pipelines[best_name]
y_prob_best = test_results[best_name]['y_prob']
print(f'Best model by test ROC-AUC: {best_name}  '
      f'(AUC = {test_results[best_name]["roc_auc"]:.4f})')

In [ ]:
fig = plot_confusion(y_test, test_results[best_name]['y_pred'],
                     title=f'Confusion matrix — {best_name}')
stamp_if_synthetic(fig)
fig.savefig('../outputs/confusion_matrix.png', dpi=150, bbox_inches='tight')

In [ ]:
fig = plot_learning_curve(best_pipe, X_train, y_train,
                          title=f'Learning curve — {best_name}')
stamp_if_synthetic(fig)
fig.savefig('../outputs/learning_curve.png', dpi=150, bbox_inches='tight')

In [ ]:
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(y_test, y_prob_best,
                                          n_bins=10, strategy='quantile')
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Perfect calibration')
ax.plot(prob_pred, prob_true, 'o-', color='#4c72b0',
        label=f'{best_name} (n_bins=10)')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Observed pass rate')
ax.set_title(f'Calibration curve — {best_name}')
ax.legend(loc='upper left')
stamp_if_synthetic(fig)
plt.tight_layout()
fig.savefig('../outputs/calibration.png', dpi=150, bbox_inches='tight')

### 8.3 Genre-ablation experiment — is this a shortcut?

If the model is essentially a dressed-up genre classifier, removing all genre features should collapse its performance. If it is picking up on broader structural patterns (era, production scale, rating), AUC should largely survive. We train an ablated pipeline identical to the best model but with `genre_*` features removed, keeping the classifier and hyperparameters identical.

In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

genre_feature_cols = [c for c in feat_cols if c.startswith('genre_')]
non_genre_cols = [c for c in feat_cols if c not in genre_feature_cols]
print(f'Removing {len(genre_feature_cols)} genre features; '
      f'{len(non_genre_cols)} features remain.')

X_train_ng = X_train[non_genre_cols]
X_test_ng  = X_test[non_genre_cols]

ablated_preproc = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                          ('scale', StandardScaler())]), NUMERIC_FEATURES),
        ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                          ('onehot', OneHotEncoder(handle_unknown='ignore',
                                                   sparse_output=False))]), CATEGORICAL_FEATURES),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)
ablated_pipe = Pipeline([
    ('preprocess', ablated_preproc),
    ('clf', clone(best_pipe.named_steps['clf'])),
])

cv_ablated = cv_score_model(ablated_pipe, X_train_ng, y_train,
                            cv_splits=5, scoring='roc_auc',
                            random_state=RANDOM_STATE)
ablated_pipe.fit(X_train_ng, y_train)
test_ablated = evaluate_on_test(ablated_pipe, X_test_ng, y_test)

cv_full     = test_results[best_name]['cv_auc_mean']
cv_full_std = test_results[best_name]['cv_auc_std']
auc_full    = test_results[best_name]['roc_auc']
auc_ablated = test_ablated['roc_auc']
drop_test   = auc_full - auc_ablated
frac_lost   = drop_test / max(auc_full - 0.5, 1e-6)

print('\n---- Genre-ablation results ----')
print(f'  FULL    CV AUC: {cv_full:.4f} ± {cv_full_std:.4f}   Test AUC: {auc_full:.4f}')
print(f'  ABLATED CV AUC: {cv_ablated["mean"]:.4f} ± {cv_ablated["std"]:.4f}   '
      f'Test AUC: {auc_ablated:.4f}')
print(f'  Test AUC drop: {drop_test:+.4f}  '
      f'({frac_lost*100:.1f}% of the signal above chance)')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
names  = ['Full model', 'No genre features']
aucs   = [auc_full, auc_ablated]
colors = ['#2ca02c', '#ff7f0e']
bars = ax.barh(names, [a - 0.5 for a in aucs], left=0.5, color=colors)
ax.set_xlim(0.5, max(0.8, auc_full + 0.05))
ax.set_xlabel('Test ROC-AUC  (0.5 = chance)')
ax.set_title('Ablation: does removing genre features collapse performance?')
for bar, auc in zip(bars, aucs):
    ax.text(auc + 0.003, bar.get_y() + bar.get_height()/2,
            f'{auc:.3f}', va='center')
ax.axvline(0.5, color='black', linestyle='--', alpha=0.5, label='Chance')
ax.legend(loc='lower right')
stamp_if_synthetic(fig)
plt.tight_layout()
fig.savefig('../outputs/ablation.png', dpi=150, bbox_inches='tight')

**How to read this result.** The '% of signal above chance' metric divides the AUC drop by the total margin above 0.5. A drop of ~70% would mean the model is mostly a genre classifier; a drop of ~15% would mean genre explains only part of the story and year, budget, and ratings carry real independent signal.

### 8.4 Per-decade performance

Does the model perform uniformly across eras, or is its signal concentrated in one period? Modern films are over-represented in the data and the Bechdel label itself has drifted in applicability. We slice test performance by decade to check.

In [ ]:
from sklearn.metrics import roc_auc_score

test_df_decade = pd.DataFrame({
    'y_true': y_test.values,
    'y_prob': y_prob_best,
    'year':   df_feat.loc[X_test.index, 'year'].values,
})
test_df_decade['decade'] = (test_df_decade['year'] // 10 * 10).astype(int)

rows = []
for decade, grp in test_df_decade.groupby('decade'):
    if grp['y_true'].nunique() < 2 or len(grp) < 30:
        continue
    rows.append({
        'decade': f'{decade}s',
        'n': len(grp),
        'pass_rate': grp['y_true'].mean(),
        'auc': roc_auc_score(grp['y_true'], grp['y_prob']),
    })
decade_perf = pd.DataFrame(rows).sort_values('decade')

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(decade_perf['decade'], decade_perf['auc'] - 0.5, bottom=0.5,
       color='#4c72b0', alpha=0.85)
ax.axhline(0.5, color='black', linestyle='--', alpha=0.4, label='Chance')
ax.axhline(auc_full, color='#d62728', linestyle='--', alpha=0.6,
           label=f'Overall ({auc_full:.3f})')
for _, r in decade_perf.iterrows():
    ax.text(r['decade'], r['auc'] + 0.006, f"n={r['n']}",
            ha='center', fontsize=8)
ax.set_ylabel('Test ROC-AUC')
ax.set_title('Per-decade test performance')
ax.legend(loc='lower right', fontsize=9)
stamp_if_synthetic(fig)
plt.tight_layout()
fig.savefig('../outputs/per_decade.png', dpi=150, bbox_inches='tight')

decade_perf.round(3)

## 9. Interpretation

### 9.1 Permutation importance

Permutation importance measures the drop in ROC-AUC when a feature's values are shuffled at test time. It is model-agnostic, reflects *generalisation* (not training fit), and does not inflate high-cardinality numeric features the way native `feature_importances_` sometimes does.

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_pipe, X_test, y_test,
    n_repeats=10, random_state=RANDOM_STATE, scoring='roc_auc', n_jobs=-1,
)
importance_df = (
    pd.DataFrame({
        'feature': X_test.columns,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std,
    })
    .sort_values('importance_mean', ascending=False)
)

fig, ax = plt.subplots(figsize=(7, 6))
top = importance_df.head(15).iloc[::-1]
ax.barh(top['feature'], top['importance_mean'],
        xerr=top['importance_std'], color='#4c72b0')
ax.set_xlabel('Permutation importance (\u0394 ROC-AUC)')
ax.set_title(f'Top 15 features \u2014 {best_name}')
stamp_if_synthetic(fig)
plt.tight_layout()
fig.savefig('../outputs/feature_importance.png', dpi=150, bbox_inches='tight')

importance_df.head(15).round(4)

### 9.2 SHAP values

Permutation importance tells us *which* features matter; SHAP tells us *how*. We run SHAP on the best tree-based model (XGBoost or RF) regardless of which model won overall, because TreeExplainer gives exact per-prediction contributions efficiently.

In [ ]:
import shap

tree_candidates = {k: v for k, v in final_pipelines.items()
                   if k.split('_')[0] in ('rf', 'xgb')}
shap_model_name = max(tree_candidates, key=lambda k: test_results[k]['roc_auc'])
shap_pipe = tree_candidates[shap_model_name]
print(f'Running SHAP on: {shap_model_name}')

pre = shap_pipe.named_steps['preprocess']
clf = shap_pipe.named_steps['clf']
X_test_prep = pre.transform(X_test)
feature_names_out = pre.get_feature_names_out()

explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test_prep)
if isinstance(shap_values, list):
    shap_values = shap_values[1]
if shap_values.ndim == 3:
    shap_values = shap_values[:, :, 1]

fig = plt.figure(figsize=(7, 5))
shap.summary_plot(shap_values, X_test_prep,
                  feature_names=feature_names_out, show=False, max_display=12)
stamp_if_synthetic(fig)
plt.tight_layout()
plt.savefig('../outputs/shap_summary.png', dpi=150, bbox_inches='tight')

### 9.3 Failure-mode analysis

Confident errors are more informative than near-threshold ones — they show where the model has *committed* to a belief that reality violates. We split them into false positives and false negatives and look for patterns.

In [ ]:
test_df = df_feat.loc[X_test.index].copy()
test_df['y_true'] = y_test.values
test_df['y_pred'] = test_results[best_name]['y_pred']
test_df['y_prob'] = y_prob_best
test_df['error'] = test_df['y_true'] != test_df['y_pred']

conf_false_pos = test_df[(test_df['y_prob'] > 0.75) &
                         (test_df['y_true'] == 0)].sort_values('y_prob', ascending=False)
conf_false_neg = test_df[(test_df['y_prob'] < 0.25) &
                         (test_df['y_true'] == 1)].sort_values('y_prob', ascending=True)

print(f'Confident false positives (predicted pass, actually failed): {len(conf_false_pos)}')
print(f'Confident false negatives (predicted fail, actually passed): {len(conf_false_neg)}')

print('\n\u2014 Top confident false positives \u2014')
display(conf_false_pos[['title', 'year', 'genres', 'y_prob']].head(5))

print('\n\u2014 Top confident false negatives \u2014')
display(conf_false_neg[['title', 'year', 'genres', 'y_prob']].head(5))

## 10. Reflection

### What worked

- The leakage-safe pipeline produced CV scores with small fold-to-fold variance, supporting the reliability of the reported means.
- Tree-based models modestly outperformed linear baselines on the full feature set, consistent with non-linear genre × era interactions.
- Permutation importance and SHAP converged on the same top features, cross-validating each other.
- The genre ablation (§8.3) gave us a direct, quantitative answer to the 'is this a shortcut?' question. The per-decade slice (§8.4) answered a similar 'where is the signal concentrated?' question.

### What didn't

- Absolute performance is modest. This is not a pipeline bug — it is a real property of the task. The Bechdel label is a coarse thresholded summary of complex film content, and metadata alone cannot closely approximate it.
- Budget and revenue are informative but ~25–30% missing. Iterative / model-based imputation did not materially outperform simple median imputation in a quick sanity test, so we did not pursue it further within our compute budget.
- Random forest underperformed both logistic regression and XGBoost — a reminder that more parameters ≠ more accuracy on modest tabular datasets.

### What we'd do next

- **Cast/crew gender composition features** from TMDb (% female cast in top-5 billed, female director 0/1, female screenwriter 0/1). These go beyond label-level summaries and plausibly carry most of the remaining headroom.
- **Plot-summary text embeddings** from a sentence-transformer, compared head-to-head against the tabular model in a tabular-only vs text-only vs fused ablation.
- **Ordinal regression** on the 0–3 Bechdel score, which uses information thrown away by the binary framing.

### Ethical considerations

A model that predicts Bechdel outcome from metadata is **not** a tool for judging films, and we would strongly object to it being deployed as one. If it were, it would entrench the genre biases it has learned: modern romantic dramas would be pre-stamped 'passes', war films 'fails', regardless of their actual dialogue. The honest framing of our work is descriptive — we document structural patterns in film production — not prescriptive. The model's errors are where it becomes most informative.

### Group collaboration

_[Fill in: who led EDA, modelling, interpretation, report. Reference GitHub commit history.]_